### 1. Install packages and get the open AI API Key
Run this once in the notebook environment.

In [1]:
%pip install -qU langchain langchain-openai pydantic

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
from getpass import getpass

if not os.environ.get("OPENAI_API_KEY"):
    api_key = os.environ["OPENAI_API_KEY"] = getpass("OpenAI API key: ")
    print(f"OpenAI API key set in environment variable OPENAI_API_KEY: {api_key[0:4]}")

OpenAI API key set in environment variable OPENAI_API_KEY: sk-p


In [3]:
# Uncomment if LangChain is not installed:
# %pip install -U langchain

from langchain.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from pprint import pprint

### 2. The mental model
A message is one unit of model context. It has a role, content, and optional metadata. Agents pass an ordered list of messages to a chat model; that list is the conversation state.

| Message | What it represents |
|---|---|
| `SystemMessage` | Instructions and context that shape the model's behavior |
| `HumanMessage` | A user's input |
| `AIMessage` | A model response; it may include tool calls and usage data |
| `ToolMessage` | The result of one tool call, sent back to the model |

In [4]:
system_msg = SystemMessage(content="You are a helpful assistant that translates English to French.")
human_msg = HumanMessage(content="Translate this sentence from English to French: 'I love programming.'")
ai_msg = AIMessage(content="J'aime programmer.")

for msg in [system_msg, human_msg, ai_msg]:
    pprint(msg.dict())

{'additional_kwargs': {},
 'content': 'You are a helpful assistant that translates English to French.',
 'id': None,
 'name': None,
 'response_metadata': {},
 'type': 'system'}
{'additional_kwargs': {},
 'content': "Translate this sentence from English to French: 'I love "
            "programming.'",
 'id': None,
 'name': None,
 'response_metadata': {},
 'type': 'human'}
{'additional_kwargs': {},
 'content': "J'aime programmer.",
 'id': None,
 'invalid_tool_calls': [],
 'name': None,
 'response_metadata': {},
 'tool_calls': [],
 'type': 'ai',
 'usage_metadata': None}


C:\Users\visha\AppData\Local\Temp\ipykernel_28240\3991178295.py:6: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  pprint(msg.dict())


### 3. Conversation history
For a multi-turn agent, append each new user message and model response to the same list. The order matters: a chat model reads the history in sequence.

In [5]:
history = [
    SystemMessage("You explain Python concepts with tiny examples."),
    HumanMessage("What is a list?"),
    AIMessage("A list is an ordered, mutable collection, for example: [1, 2, 3]."),
    HumanMessage("How do I add an item?"),
]

# In an agent, the next model call receives all of this context.
for index, message in enumerate(history, start=1):
    print(f"Turn {index} — {message.type}: {message.content}")

Turn 1 — system: You explain Python concepts with tiny examples.
Turn 2 — human: What is a list?
Turn 3 — ai: A list is an ordered, mutable collection, for example: [1, 2, 3].
Turn 4 — human: How do I add an item?


### 4. Metadata and useful fields
Messages can carry IDs, names, provider response metadata, and token usage. A provider usually supplies AI-message metadata after an actual model call; the example below creates small values locally so it runs anywhere.

In [6]:
named_user = HumanMessage(
    content="Please explain recursion.",
    name="learner",
    id="user-001",
)

sample_response = AIMessage(
    content="Recursion is when a function calls itself until it reaches a base case.",
    id="ai-001",
    usage_metadata={"input_tokens": 12, "output_tokens": 16, "total_tokens": 28},
)

print(named_user.id, named_user.name)
pprint(sample_response.usage_metadata)
print("Text shortcut:", sample_response.text)

user-001 learner
{'input_tokens': 12, 'output_tokens': 16, 'total_tokens': 28}
Text shortcut: Recursion is when a function calls itself until it reaches a base case.


### 5. Combination of Model initialization, invoke and Messages

In [8]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model_name="gpt-4.1-mini", temperature=0)

system_msg = SystemMessage(content="You are a helpful assistant that translates English to French.")
human_msg = HumanMessage(content="Translate this sentence from English to French: 'I love programming.'")

model_response = model.invoke([system_msg, human_msg])
print(model_response.content)

J'aime la programmation.
